# Data and Analysis Archiving

This notebook packages all components necessary to fully recapitulate an analysis into a timestamped archive folder. This includes:
- Raw data and legacy metadata files embedded in raw folders
- Central metadata YAML files from `data/metadata/`
- Processed/compiled data
- Analysis results from `data/results/`
- All notebooks and scripts
- Configuration files (requirements.txt, .gitignore, etc.)
- Python modules (.py files)

The archive is self-contained and can be shared or stored for reproducibility.

## Setup and Configuration

Import required libraries and define the archive destination. Archives are saved to `data/archive/` with timestamps.

In [1]:
from pathlib import Path
from datetime import datetime
import shutil
import json

# Project root directory
project_root = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()

# Archive destination
archive_base = project_root / "data" / "archive"
archive_base.mkdir(parents=True, exist_ok=True)

# Generate timestamp for this archive
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_name = f"celegans_analysis_{timestamp}"
archive_dir = archive_base / archive_name

print(f"Project root: {project_root}")
print(f"Archive will be created at: {archive_dir}")

Project root: c:\Users\MBF\celegans-volume-analysis
Archive will be created at: c:\Users\MBF\celegans-volume-analysis\data\archive\celegans_analysis_20260604_112333


## Define Archive Contents

Specify all files and folders to include in the archive for full reproducibility.

In [2]:
def gather_archive_contents(root_dir: Path):
    """
    Gather all files and folders needed for a complete archive.
    Returns a dict mapping source paths to relative archive paths.
    """
    archive_contents = {}
    
    # 1. Raw data (all genotype folders with config.yaml and video data)
    raw_data_dir = root_dir / "data" / "raw"
    if raw_data_dir.exists():
        for item in raw_data_dir.rglob("*"):
            if item.is_file():
                rel_path = item.relative_to(root_dir)
                archive_contents[item] = rel_path
    
    # 2. Centralized metadata YAML files
    metadata_dir = root_dir / "data" / "metadata"
    if metadata_dir.exists():
        for item in metadata_dir.rglob("*"):
            if item.is_file():
                rel_path = item.relative_to(root_dir)
                archive_contents[item] = rel_path
    
    # 3. Processed/compiled data
    processed_dir = root_dir / "data" / "processed"
    if processed_dir.exists():
        for item in processed_dir.rglob("*"):
            if item.is_file():
                rel_path = item.relative_to(root_dir)
                archive_contents[item] = rel_path
    
    # 4. Analysis results
    results_dir = root_dir / "data" / "results"
    if results_dir.exists():
        for item in results_dir.rglob("*"):
            if item.is_file():
                rel_path = item.relative_to(root_dir)
                archive_contents[item] = rel_path
    
    # 5. All notebooks
    notebooks_dir = root_dir / "Notebooks"
    if notebooks_dir.exists():
        for nb in notebooks_dir.glob("*.ipynb"):
            rel_path = nb.relative_to(root_dir)
            archive_contents[nb] = rel_path
    
    # 6. All scripts
    scripts_dir = root_dir / "scripts"
    if scripts_dir.exists():
        for script in scripts_dir.rglob("*.py"):
            rel_path = script.relative_to(root_dir)
            archive_contents[script] = rel_path
    
    # 7. Root-level Python files
    for py_file in root_dir.glob("*.py"):
        rel_path = py_file.relative_to(root_dir)
        archive_contents[py_file] = rel_path
    
    # 8. Configuration files
    config_files = [
        "requirements.txt",
        ".gitignore",
        "README.md",
        "setup.py",
        "pyproject.toml",
        "setup.cfg"
    ]
    for cfg in config_files:
        cfg_path = root_dir / cfg
        if cfg_path.exists():
            rel_path = cfg_path.relative_to(root_dir)
            archive_contents[cfg_path] = rel_path
    
    return archive_contents


# Gather all content
contents = gather_archive_contents(project_root)
print(f"Found {len(contents)} files to archive")
print(f"\nBreakdown by directory:")

# Count by top-level directory
from collections import Counter
dir_counts = Counter([str(path).split('/')[0] if '/' in str(path) or '\\' in str(path) else 'root' 
                      for path in contents.values()])
for dir_name, count in sorted(dir_counts.items()):
    print(f"  {dir_name}: {count} files")

# Show data folder breakdown so metadata inclusion is explicit.
data_subdir_counts = Counter(
    Path(path).parts[1]
    for path in contents.values()
    if len(Path(path).parts) > 1 and Path(path).parts[0] == "data"
)
print("\nData subfolder breakdown:")
for subdir_name, count in sorted(data_subdir_counts.items()):
    print(f"  data/{subdir_name}: {count} files")

Found 150 files to archive

Breakdown by directory:
  Notebooks\01_compile_genotype_data.ipynb: 1 files
  Notebooks\02_celegans_volume_analysis.ipynb: 1 files
  Notebooks\03_archiving.ipynb: 1 files
  data\metadata\N2_1050.yaml: 1 files
  data\metadata\N2_450.yaml: 1 files
  data\processed\.gitkeep: 1 files
  data\processed\260604_Wormlab_exports_N2_1050_compiled_area.csv: 1 files
  data\processed\260604_Wormlab_exports_N2_1050_compiled_bending_angle.csv: 1 files
  data\processed\260604_Wormlab_exports_N2_1050_compiled_fit.csv: 1 files
  data\processed\260604_Wormlab_exports_N2_1050_compiled_length.csv: 1 files
  data\processed\260604_Wormlab_exports_N2_450_compiled_area.csv: 1 files
  data\processed\260604_Wormlab_exports_N2_450_compiled_bending_angle.csv: 1 files
  data\processed\260604_Wormlab_exports_N2_450_compiled_fit.csv: 1 files
  data\processed\260604_Wormlab_exports_N2_450_compiled_length.csv: 1 files
  data\processed\compilation_summary.csv: 1 files
  data\processed\genotype

## Create Archive and Copy Files

Create the timestamped archive directory and copy all files while preserving the directory structure.

In [3]:
def create_archive(archive_path: Path, contents: dict, create_manifest: bool = True):
    """
    Copy all files to archive directory, preserving structure.
    
    Args:
        archive_path: Destination archive directory
        contents: Dict mapping source paths to relative archive paths
        create_manifest: Whether to create a JSON manifest of archived files
    
    Returns:
        Tuple of (success_count, total_count)
    """
    archive_path.mkdir(parents=True, exist_ok=True)
    
    success_count = 0
    failed_files = []
    manifest_data = {
        "archive_name": archive_path.name,
        "created": datetime.now().isoformat(),
        "total_files": len(contents),
        "files": []
    }
    
    print(f"Creating archive at: {archive_path}")
    print(f"Copying {len(contents)} files...")
    
    for source, rel_dest in contents.items():
        dest = archive_path / rel_dest
        
        try:
            # Create parent directories if needed
            dest.parent.mkdir(parents=True, exist_ok=True)
            
            # Copy file
            shutil.copy2(source, dest)
            success_count += 1
            
            # Add to manifest
            manifest_data["files"].append({
                "path": str(rel_dest),
                "size_bytes": source.stat().st_size,
                "modified": datetime.fromtimestamp(source.stat().st_mtime).isoformat()
            })
            
            # Progress indicator
            if success_count % 10 == 0:
                print(f"  Copied {success_count}/{len(contents)} files...", end='\r')
                
        except Exception as e:
            failed_files.append((source, str(e)))
            print(f"\n  Warning: Failed to copy {source}: {e}")
    
    print(f"\nCompleted: {success_count}/{len(contents)} files copied successfully")
    
    # Save manifest
    if create_manifest:
        manifest_path = archive_path / "MANIFEST.json"
        with open(manifest_path, 'w') as f:
            json.dump(manifest_data, f, indent=2)
        print(f"Manifest saved to: {manifest_path}")
    
    # Report any failures
    if failed_files:
        print(f"\n⚠️  {len(failed_files)} files failed to copy:")
        for path, error in failed_files[:5]:  # Show first 5
            print(f"  - {path}: {error}")
        if len(failed_files) > 5:
            print(f"  ... and {len(failed_files) - 5} more")
    
    return success_count, len(contents)


# Execute archiving
success, total = create_archive(archive_dir, contents, create_manifest=True)
print(f"\n{'='*60}")
print(f"✅ Archive created successfully!")
print(f"   Location: {archive_dir}")
print(f"   Files: {success}/{total}")
print(f"   Size: {sum(f.stat().st_size for f in contents.keys()) / 1024 / 1024:.2f} MB")
print(f"{'='*60}")

Creating archive at: c:\Users\MBF\celegans-volume-analysis\data\archive\celegans_analysis_20260604_112333
Copying 150 files...
  Copied 150/150 files...
Completed: 150/150 files copied successfully
Manifest saved to: c:\Users\MBF\celegans-volume-analysis\data\archive\celegans_analysis_20260604_112333\MANIFEST.json

✅ Archive created successfully!
   Location: c:\Users\MBF\celegans-volume-analysis\data\archive\celegans_analysis_20260604_112333
   Files: 150/150
   Size: 163.82 MB


## Verify Archive Contents

Display a summary of what was archived and verify the archive structure.

In [4]:
import pandas as pd

# Load and display manifest
manifest_path = archive_dir / "MANIFEST.json"
if manifest_path.exists():
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    
    print(f"Archive: {manifest['archive_name']}")
    print(f"Created: {manifest['created']}")
    print(f"Total files: {manifest['total_files']}")
    print("\n" + "="*80)
    
    # Create summary DataFrame
    files_df = pd.DataFrame(manifest['files'])
    files_df['size_kb'] = files_df['size_bytes'] / 1024
    files_df['directory'] = files_df['path'].apply(lambda x: str(Path(x).parts[0]) if len(Path(x).parts) > 0 else 'root')
    
    # Summary by directory
    print("\nArchive contents by directory:")
    summary = files_df.groupby('directory').agg({
        'path': 'count',
        'size_kb': 'sum'
    }).rename(columns={'path': 'file_count', 'size_kb': 'total_size_kb'})
    summary['total_size_mb'] = summary['total_size_kb'] / 1024
    summary = summary.sort_values('total_size_mb', ascending=False)
    
    display(summary[['file_count', 'total_size_mb']].round(2))
    
    print(f"\n✅ Archive is ready for distribution or long-term storage")
    print(f"   Path: {archive_dir}")
else:
    print("Warning: Manifest file not found!")

Archive: celegans_analysis_20260604_112333
Created: 2026-06-04T11:23:40.456461
Total files: 150


Archive contents by directory:


,file_count,total_size_mb
directory,,
data,138,153.19
Notebooks,3,10.60
README.md,1,0.01
run_stats.py,1,0.01
scripts,2,0.00
add_cell.py,1,0.00
wt_pairwise_tests.py,1,0.00
.gitignore,1,0.00
verify_cell.py,1,0.00



✅ Archive is ready for distribution or long-term storage
   Path: c:\Users\MBF\celegans-volume-analysis\data\archive\celegans_analysis_20260604_112333


## Archive Summary

This notebook creates a timestamped archive folder containing all files needed to reproduce your analysis and preserve outputs.

**What's included in the archive:**
- All raw data from `data/raw/` (including legacy config.yaml metadata files)
- All central metadata YAML files from `data/metadata/`
- All processed/compiled data from `data/processed/`
- All analysis outputs from `data/results/`
- All Jupyter notebooks from `Notebooks/`
- All Python scripts from `scripts/` and the project root
- Key project configuration files such as `requirements.txt`, `.gitignore`, and `README.md`

**Archive structure:**
- Archives are saved to `data/archive/` with timestamp: `celegans_analysis_YYYYMMDD_HHMMSS/`
- Original directory structure is preserved inside the archive
- A `MANIFEST.json` file is included with metadata about all archived files

**Use cases:**
- **Reproducibility**: Preserve the exact state of inputs, code, and outputs used for an analysis
- **Sharing**: Send a self-contained analysis package to collaborators
- **Long-term storage**: Archive completed analyses with all necessary files

**To restore an archive:**
Simply extract the archive contents to recreate the full analysis environment, install dependencies from requirements.txt, and re-run the notebooks.

## Optional Workspace Cleanup for the Next Dataset

After archiving, you may want to clear dataset-specific files so the workspace is ready for a new run.

**Recommended to clear before a new dataset:**
- `data/raw/`
- `data/metadata/`
- `data/processed/`
- `data/results/`

**Usually keep:**
- `data/archive/`
- notebooks, scripts, and project configuration files

**Optional extras to clear:**
- `__pycache__/` directories
- `.ipynb_checkpoints/` directories
- `.pytest_cache/` if present

The cleanup cell below is intentionally guarded. It defaults to a dry run and will only delete files after you explicitly enable it and confirm the action.

In [5]:
from pathlib import Path
import shutil

# Safety gates: review the dry-run output first.
CLEANUP_ENABLED = True
CLEANUP_CONFIRM_TEXT = "DELETE_ARCHIVED_DATA"
REQUIRED_CONFIRM_TEXT = "DELETE_ARCHIVED_DATA"

# Core dataset-specific directories to clear for the next run.
CLEANUP_TARGETS = [
    project_root / "data" / "raw",
    project_root / "data" / "metadata",
    project_root / "data" / "processed",
    project_root / "data" / "results",
]

# Optional extras that are safe to remove if you want a cleaner workspace.
REMOVE_PYTHON_CACHES = True
REMOVE_NOTEBOOK_CHECKPOINTS = True
REMOVE_PYTEST_CACHE = True
PRESERVE_GITKEEP_FILES = True

archive_base = project_root / "data" / "archive"
archive_dir = archive_dir if 'archive_dir' in globals() else None

def iter_directory_contents(path: Path):
    if not path.exists():
        return []
    return sorted(path.iterdir(), key=lambda p: (p.is_file(), str(p).lower()))

def should_skip_path(path: Path):
    if PRESERVE_GITKEEP_FILES and path.name == '.gitkeep':
        return True
    return False

def collect_cleanup_plan():
    planned = []

    for target in CLEANUP_TARGETS:
        target = Path(target)
        if not target.exists():
            planned.append({
                'path': target,
                'kind': 'missing',
                'action': 'skip',
            })
            continue

        for item in iter_directory_contents(target):
            if should_skip_path(item):
                continue
            planned.append({
                'path': item,
                'kind': 'dir' if item.is_dir() else 'file',
                'action': 'delete',
            })

    if REMOVE_PYTHON_CACHES:
        for cache_dir in project_root.rglob('__pycache__'):
            if archive_base in cache_dir.parents or cache_dir == archive_base:
                continue
            planned.append({
                'path': cache_dir,
                'kind': 'dir',
                'action': 'delete_optional',
            })

    if REMOVE_NOTEBOOK_CHECKPOINTS:
        for checkpoint_dir in project_root.rglob('.ipynb_checkpoints'):
            if archive_base in checkpoint_dir.parents or checkpoint_dir == archive_base:
                continue
            planned.append({
                'path': checkpoint_dir,
                'kind': 'dir',
                'action': 'delete_optional',
            })

    if REMOVE_PYTEST_CACHE:
        for pytest_cache in project_root.rglob('.pytest_cache'):
            if archive_base in pytest_cache.parents or pytest_cache == archive_base:
                continue
            planned.append({
                'path': pytest_cache,
                'kind': 'dir',
                'action': 'delete_optional',
            })

    # Deduplicate while preserving order.
    deduped = []
    seen = set()
    for item in planned:
        key = str(item['path']).lower()
        if key in seen:
            continue
        seen.add(key)
        deduped.append(item)
    return deduped

def summarize_cleanup_plan(plan):
    delete_count = sum(1 for item in plan if item['action'].startswith('delete'))
    file_count = sum(1 for item in plan if item['kind'] == 'file' and item['action'].startswith('delete'))
    dir_count = sum(1 for item in plan if item['kind'] == 'dir' and item['action'].startswith('delete'))
    print('Cleanup plan summary:')
    print(f'  Entries to remove: {delete_count}')
    print(f'  Files: {file_count}')
    print(f'  Directories: {dir_count}')
    print('')
    print('Targets:')
    for target in CLEANUP_TARGETS:
        status = 'exists' if Path(target).exists() else 'missing'
        print(f'  - {target} [{status}]')
    if PRESERVE_GITKEEP_FILES:
        print('  - preserving .gitkeep placeholder files')
    print('')
    print('Planned removals (first 25):')
    preview = [item for item in plan if item['action'].startswith('delete')]
    for item in preview[:25]:
        print(f"  - {item['action']}: {item['path']}")
    if len(preview) > 25:
        print(f'  ... and {len(preview) - 25} more')

def execute_cleanup(plan):
    deleted = []
    failed = []

    for item in plan:
        if not item['action'].startswith('delete'):
            continue

        path = Path(item['path'])
        if not path.exists():
            continue

        try:
            if path.is_dir():
                shutil.rmtree(path)
            else:
                path.unlink()
            deleted.append(path)
        except Exception as exc:
            failed.append((path, str(exc)))

    # Recreate the core directories so the next dataset has a clean structure.
    for target in CLEANUP_TARGETS:
        Path(target).mkdir(parents=True, exist_ok=True)

    return deleted, failed

cleanup_plan = collect_cleanup_plan()
summarize_cleanup_plan(cleanup_plan)

if not CLEANUP_ENABLED:
    print('')
    print('Dry run only. No files were deleted.')
    print(f'Set CLEANUP_ENABLED = True and CLEANUP_CONFIRM_TEXT = "{REQUIRED_CONFIRM_TEXT}" to perform deletion.')
elif CLEANUP_CONFIRM_TEXT != REQUIRED_CONFIRM_TEXT:
    raise ValueError(
        f'Cleanup confirmation text mismatch. Set CLEANUP_CONFIRM_TEXT = "{REQUIRED_CONFIRM_TEXT}" to continue.'
    )
else:
    deleted_paths, failed_paths = execute_cleanup(cleanup_plan)
    print('')
    print(f'Deleted {len(deleted_paths)} paths.')
    if failed_paths:
        print(f'{len(failed_paths)} paths failed to delete:')
        for path, error in failed_paths[:10]:
            print(f'  - {path}: {error}')
        if len(failed_paths) > 10:
            print(f'  ... and {len(failed_paths) - 10} more')
    else:
        print('Cleanup completed without deletion errors.')

Cleanup plan summary:
  Entries to remove: 366
  Files: 115
  Directories: 251

Targets:
  - c:\Users\MBF\celegans-volume-analysis\data\raw [exists]
  - c:\Users\MBF\celegans-volume-analysis\data\metadata [exists]
  - c:\Users\MBF\celegans-volume-analysis\data\processed [exists]
  - c:\Users\MBF\celegans-volume-analysis\data\results [exists]
  - preserving .gitkeep placeholder files

Planned removals (first 25):
  - delete: c:\Users\MBF\celegans-volume-analysis\data\raw\260604
  - delete: c:\Users\MBF\celegans-volume-analysis\data\metadata\N2_1050.yaml
  - delete: c:\Users\MBF\celegans-volume-analysis\data\metadata\N2_450.yaml
  - delete: c:\Users\MBF\celegans-volume-analysis\data\processed\260604_Wormlab_exports_N2_1050_compiled_area.csv
  - delete: c:\Users\MBF\celegans-volume-analysis\data\processed\260604_Wormlab_exports_N2_1050_compiled_bending_angle.csv
  - delete: c:\Users\MBF\celegans-volume-analysis\data\processed\260604_Wormlab_exports_N2_1050_compiled_fit.csv
  - delete: c:\

## Restore Notebooks From an Archive for Reanalysis

Use this cell to replace the active notebooks with the archived versions so reanalysis uses the exact notebook state from the archive.

Behavior:
- Copies all `.ipynb` files from `data/archive/<archive_name>/Notebooks/` into the active `Notebooks/` folder
- Optionally removes active notebooks that are not present in the archive (exact sync)
- Creates a timestamped backup of current active notebooks before overwrite

In [8]:
from pathlib import Path
from datetime import datetime
import shutil

# Ensure this cell is runnable independently.
project_root = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()

# Safety gates (review paths before enabling).
RESTORE_NOTEBOOKS_ENABLED = True
RESTORE_CONFIRM_TEXT = "RESTORE_ARCHIVED_NOTEBOOKS"
REQUIRED_RESTORE_CONFIRM_TEXT = "RESTORE_ARCHIVED_NOTEBOOKS"

# Select archive to restore from.
# Option 1: set an explicit archive folder name, e.g. "celegans_analysis_20260413_160152"
RESTORE_ARCHIVE_NAME = None

# Option 2: if None, auto-select the most recently modified archive folder.
AUTO_SELECT_LATEST_ARCHIVE = True

# If True, remove active notebooks not present in the archive (exact notebook sync).
SYNC_EXACT_NOTEBOOK_SET = True

# If True, back up active notebooks before replacement.
BACKUP_ACTIVE_NOTEBOOKS = False

archive_base = project_root / "data" / "archive"
active_notebooks_dir = project_root / "Notebooks"
active_notebooks_dir.mkdir(parents=True, exist_ok=True)

if not archive_base.exists():
    raise ValueError(f"Archive base does not exist: {archive_base}")

def choose_archive_dir(base_dir: Path, explicit_name=None, auto_latest=True):
    if explicit_name:
        candidate = base_dir / explicit_name
        if not candidate.exists() or not candidate.is_dir():
            raise ValueError(f"Requested archive folder not found: {candidate}")
        return candidate

    if auto_latest:
        candidates = [p for p in base_dir.iterdir() if p.is_dir()]
        if not candidates:
            raise ValueError(f"No archive folders found under: {base_dir}")
        candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        return candidates[0]

    raise ValueError("No archive selected. Set RESTORE_ARCHIVE_NAME or enable AUTO_SELECT_LATEST_ARCHIVE.")

selected_archive_dir = choose_archive_dir(
    archive_base,
    explicit_name=RESTORE_ARCHIVE_NAME,
    auto_latest=AUTO_SELECT_LATEST_ARCHIVE,
 )

archived_notebooks_dir = selected_archive_dir / "Notebooks"
if not archived_notebooks_dir.exists() or not archived_notebooks_dir.is_dir():
    raise ValueError(f"Archive does not contain a Notebooks folder: {archived_notebooks_dir}")

archived_notebooks = sorted(archived_notebooks_dir.glob("*.ipynb"))
if not archived_notebooks:
    raise ValueError(f"No notebook files found in archive Notebooks folder: {archived_notebooks_dir}")

active_notebooks = sorted(active_notebooks_dir.glob("*.ipynb"))
archived_names = {p.name for p in archived_notebooks}
active_names = {p.name for p in active_notebooks}
extra_active = sorted(active_names - archived_names)

print("Restore plan:")
print(f"  Source archive: {selected_archive_dir}")
print(f"  Archived notebooks found: {len(archived_notebooks)}")
for nb in archived_notebooks:
    print(f"    - {nb.name}")
print(f"  Active notebooks currently present: {len(active_notebooks)}")
if extra_active:
    print("  Active notebooks not present in archive:")
    for name in extra_active:
        print(f"    - {name}")
else:
    print("  No extra active notebooks detected.")
print("")
print(f"RESTORE_NOTEBOOKS_ENABLED={RESTORE_NOTEBOOKS_ENABLED}")
print(f"SYNC_EXACT_NOTEBOOK_SET={SYNC_EXACT_NOTEBOOK_SET}")
print(f"BACKUP_ACTIVE_NOTEBOOKS={BACKUP_ACTIVE_NOTEBOOKS}")

if not RESTORE_NOTEBOOKS_ENABLED:
    print("")
    print("Dry run only. No files were changed.")
    print(f"Set RESTORE_NOTEBOOKS_ENABLED = True and RESTORE_CONFIRM_TEXT = '{REQUIRED_RESTORE_CONFIRM_TEXT}' to restore.")
elif RESTORE_CONFIRM_TEXT != REQUIRED_RESTORE_CONFIRM_TEXT:
    raise ValueError(
        f"Restore confirmation text mismatch. Set RESTORE_CONFIRM_TEXT = '{REQUIRED_RESTORE_CONFIRM_TEXT}' to continue."
    )
else:
    backup_dir = None
    if BACKUP_ACTIVE_NOTEBOOKS and active_notebooks:
        backup_root = archive_base / "notebook_restore_backups"
        backup_root.mkdir(parents=True, exist_ok=True)
        backup_dir = backup_root / f"active_notebooks_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        backup_dir.mkdir(parents=True, exist_ok=True)
        for nb in active_notebooks:
            shutil.copy2(nb, backup_dir / nb.name)
        print(f"Backed up {len(active_notebooks)} active notebook(s) to: {backup_dir}")

    copied = []
    for src_nb in archived_notebooks:
        dest_nb = active_notebooks_dir / src_nb.name
        shutil.copy2(src_nb, dest_nb)
        copied.append(dest_nb.name)

    removed = []
    if SYNC_EXACT_NOTEBOOK_SET:
        for extra_name in extra_active:
            extra_path = active_notebooks_dir / extra_name
            if extra_path.exists():
                extra_path.unlink()
                removed.append(extra_name)

    print("")
    print("Notebook restore completed.")
    print(f"  Restored notebooks: {len(copied)}")
    if removed:
        print(f"  Removed notebooks not in archive: {len(removed)}")
        for name in removed:
            print(f"    - {name}")
    else:
        print("  Removed notebooks not in archive: 0")
    print(f"  Active notebooks directory: {active_notebooks_dir}")
    if backup_dir is not None:
        print(f"  Backup location: {backup_dir}")

Restore plan:
  Source archive: c:\Users\MBF\celegans-volume-analysis\data\archive\celegans_analysis_20260604_112333
  Archived notebooks found: 3
    - 01_compile_genotype_data.ipynb
    - 02_celegans_volume_analysis.ipynb
    - 03_archiving.ipynb
  Active notebooks currently present: 3
  No extra active notebooks detected.

RESTORE_NOTEBOOKS_ENABLED=True
SYNC_EXACT_NOTEBOOK_SET=True
BACKUP_ACTIVE_NOTEBOOKS=False

Notebook restore completed.
  Restored notebooks: 3
  Removed notebooks not in archive: 0
  Active notebooks directory: c:\Users\MBF\celegans-volume-analysis\Notebooks


In [9]:
from pathlib import Path
from datetime import datetime
import shutil

# Restore data folders from a selected archive.
DATA_RESTORE_ENABLED = True
DATA_RESTORE_CONFIRM_TEXT = "RESTORE_ARCHIVED_DATA_FOLDERS"
REQUIRED_DATA_RESTORE_CONFIRM_TEXT = "RESTORE_ARCHIVED_DATA_FOLDERS"

DATA_RESTORE_ARCHIVE_NAME = "celegans_analysis_20260604_112333"

# Data folders to restore from archive root into project root.
DATA_FOLDERS_TO_RESTORE = [
    Path("data/raw"),
    Path("data/metadata"),
    Path("data/processed"),
    Path("data/results"),
]

# If True, remove files/folders in destination that are not in archive for those data folders.
DATA_RESTORE_EXACT_SYNC = True

# If True, back up destination data folders before restore.
BACKUP_DATA_BEFORE_RESTORE = True

archive_base = project_root / "data" / "archive"
selected_archive_dir = archive_base / DATA_RESTORE_ARCHIVE_NAME

if not selected_archive_dir.exists() or not selected_archive_dir.is_dir():
    raise ValueError(f"Requested archive folder not found: {selected_archive_dir}")

backup_dir = None
if BACKUP_DATA_BEFORE_RESTORE:
    backup_root = archive_base / "notebook_restore_backups"
    backup_root.mkdir(parents=True, exist_ok=True)
    backup_dir = backup_root / f"active_data_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

def collect_files_and_dirs(base_path: Path):
    files = set()
    dirs = set()
    if not base_path.exists():
        return files, dirs
    for p in base_path.rglob("*"):
        rel = p.relative_to(base_path)
        if p.is_dir():
            dirs.add(rel)
        else:
            files.add(rel)
    return files, dirs

print("Data restore plan:")
print(f"  Source archive: {selected_archive_dir}")
print(f"  Exact sync: {DATA_RESTORE_EXACT_SYNC}")
print(f"  Backup enabled: {BACKUP_DATA_BEFORE_RESTORE}")
print("  Folders:")
for rel_folder in DATA_FOLDERS_TO_RESTORE:
    print(f"    - {rel_folder}")

if not DATA_RESTORE_ENABLED:
    print("")
    print("Dry run only. No files were changed.")
    print(f"Set DATA_RESTORE_ENABLED = True and DATA_RESTORE_CONFIRM_TEXT = '{REQUIRED_DATA_RESTORE_CONFIRM_TEXT}' to restore data folders.")
elif DATA_RESTORE_CONFIRM_TEXT != REQUIRED_DATA_RESTORE_CONFIRM_TEXT:
    raise ValueError(
        f"Data restore confirmation mismatch. Set DATA_RESTORE_CONFIRM_TEXT = '{REQUIRED_DATA_RESTORE_CONFIRM_TEXT}' to continue."
    )
else:
    restored_files = 0
    removed_files = 0
    removed_dirs = 0
    backup_count = 0

    if backup_dir is not None:
        backup_dir.mkdir(parents=True, exist_ok=True)

    for rel_folder in DATA_FOLDERS_TO_RESTORE:
        src_root = selected_archive_dir / rel_folder
        dst_root = project_root / rel_folder

        if not src_root.exists() or not src_root.is_dir():
            print(f"Skipping missing folder in archive: {src_root}")
            continue

        # Backup current destination folder before restoring.
        if backup_dir is not None and dst_root.exists():
            backup_target = backup_dir / rel_folder
            backup_target.parent.mkdir(parents=True, exist_ok=True)
            if backup_target.exists():
                shutil.rmtree(backup_target)
            shutil.copytree(dst_root, backup_target)
            backup_count += 1

        dst_root.mkdir(parents=True, exist_ok=True)

        src_files, src_dirs = collect_files_and_dirs(src_root)
        dst_files, dst_dirs = collect_files_and_dirs(dst_root)

        # Copy/overwrite files from archive.
        for rel_file in src_files:
            src_file = src_root / rel_file
            dst_file = dst_root / rel_file
            dst_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_file, dst_file)
            restored_files += 1

        # Optional exact sync removal for files/dirs not in archive.
        if DATA_RESTORE_EXACT_SYNC:
            extra_files = sorted(dst_files - src_files)
            for rel_file in extra_files:
                p = dst_root / rel_file
                if p.exists() and p.is_file():
                    p.unlink()
                    removed_files += 1

            extra_dirs = sorted(dst_dirs - src_dirs, key=lambda x: len(x.parts), reverse=True)
            for rel_dir in extra_dirs:
                p = dst_root / rel_dir
                if p.exists() and p.is_dir():
                    try:
                        p.rmdir()
                        removed_dirs += 1
                    except OSError:
                        # Directory not empty; leave it in place.
                        pass

        print(f"Restored folder: {rel_folder}")

    print("")
    print("Data restore completed.")
    print(f"  Files restored (copied/overwritten): {restored_files}")
    print(f"  Extra files removed: {removed_files}")
    print(f"  Extra empty dirs removed: {removed_dirs}")
    if backup_dir is not None:
        if backup_count > 0:
            print(f"  Backup location: {backup_dir}")
        else:
            print("  Backup requested, but no existing destination folders needed backup.")

Data restore plan:
  Source archive: c:\Users\MBF\celegans-volume-analysis\data\archive\celegans_analysis_20260604_112333
  Exact sync: True
  Backup enabled: True
  Folders:
    - data\raw
    - data\metadata
    - data\processed
    - data\results
Restored folder: data\raw
Restored folder: data\metadata
Restored folder: data\processed
Restored folder: data\results

Data restore completed.
  Files restored (copied/overwritten): 138
  Extra files removed: 0
  Extra empty dirs removed: 0
  Backup location: c:\Users\MBF\celegans-volume-analysis\data\archive\notebook_restore_backups\active_data_backup_20260604_115931


In [ ]:
from pathlib import Path
import ast
import pandas as pd
import yaml

# Rebuild central metadata YAML files from processed genotype metadata table.
REBUILD_METADATA_ENABLED = True
REBUILD_METADATA_CONFIRM_TEXT = "REBUILD_METADATA_FROM_GENOTYPE_CSV"
REQUIRED_REBUILD_METADATA_CONFIRM_TEXT = "REBUILD_METADATA_FROM_GENOTYPE_CSV"
OVERWRITE_EXISTING_METADATA = True

project_root = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
metadata_csv_path = project_root / "data" / "processed" / "genotype_metadata.csv"
metadata_out_dir = project_root / "data" / "metadata"
metadata_out_dir.mkdir(parents=True, exist_ok=True)

def parse_maybe_dict(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    text = str(value).strip()
    if not text:
        return {}
    try:
        parsed = ast.literal_eval(text)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}

def to_native(obj):
    if isinstance(obj, dict):
        return {str(k): to_native(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_native(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_native(v) for v in obj]
    if pd.isna(obj):
        return None
    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            pass
    return obj

if not metadata_csv_path.exists():
    raise FileNotFoundError(f"Missing metadata table: {metadata_csv_path}")

if not REBUILD_METADATA_ENABLED:
    print("Dry run only. No metadata files were written.")
    print(
        f"Set REBUILD_METADATA_ENABLED = True and REBUILD_METADATA_CONFIRM_TEXT = '{REQUIRED_REBUILD_METADATA_CONFIRM_TEXT}' to rebuild metadata."
    )
elif REBUILD_METADATA_CONFIRM_TEXT != REQUIRED_REBUILD_METADATA_CONFIRM_TEXT:
    raise ValueError(
        f"Confirmation mismatch. Set REBUILD_METADATA_CONFIRM_TEXT = '{REQUIRED_REBUILD_METADATA_CONFIRM_TEXT}' to continue."
    )
else:
    df = pd.read_csv(metadata_csv_path)
    required_cols = ["key"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Metadata CSV missing required columns: {missing_cols}")

    created = []
    skipped = []

    for key, g in df.groupby("key", dropna=True):
        row = g.iloc[0]

        out_path = metadata_out_dir / f"{key}.yaml"
        if out_path.exists() and not OVERWRITE_EXISTING_METADATA:
            skipped.append(out_path.name)
            continue

        doc = {
            "assay": row.get("assay") if pd.notna(row.get("assay")) else "volume",
            "experiment_id": row.get("experiment_id") if pd.notna(row.get("experiment_id")) else None,
            "assay_date": row.get("assay_date") if pd.notna(row.get("assay_date")) else None,
            "analysis_date": row.get("analysis_date") if pd.notna(row.get("analysis_date")) else None,
            "genotype": row.get("genotype") if pd.notna(row.get("genotype")) else None,
            "strain": row.get("strain") if pd.notna(row.get("strain")) else None,
            "background": row.get("background") if pd.notna(row.get("background")) else None,
            "outcrosses": row.get("outcrosses") if pd.notna(row.get("outcrosses")) else None,
            "age": row.get("age") if pd.notna(row.get("age")) else None,
            "NaCl_concentration_mM": row.get("NaCl_concentration_mM") if pd.notna(row.get("NaCl_concentration_mM")) else None,
            "buffer": row.get("buffer") if pd.notna(row.get("buffer")) else None,
            "experimenter": row.get("experimenter") if pd.notna(row.get("experimenter")) else None,
            "lab": row.get("lab") if pd.notna(row.get("lab")) else None,
            "arena": parse_maybe_dict(row.get("arena")),
            "imaging": parse_maybe_dict(row.get("imaging")),
            "wormlab": parse_maybe_dict(row.get("wormlab")),
            "notes": row.get("notes") if pd.notna(row.get("notes")) else None,
            "videos": parse_maybe_dict(row.get("videos")),
        }

        with open(out_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(to_native(doc), f, sort_keys=False, allow_unicode=False)
        created.append(out_path.name)

    print(f"Rebuilt metadata files: {len(created)}")
    if created:
        print("First 20 created files:")
        for name in created[:20]:
            print(f"  - {name}")
        if len(created) > 20:
            print(f"  ... and {len(created) - 20} more")
    if skipped:
        print(f"Skipped existing files: {len(skipped)}")